In [1]:
import pandas as pd
import json
from datetime import datetime, timezone, UTC

In [2]:
# The number is less then actual uniue tweets in filtered notes
# This means those tweets were not scraped because of they unavailability or error in scraping
with open('Outputs/scraped_tweets_processed.json', 'r') as f:
    tweets = json.load(f)

print(f"Total unique tweets scraped from filtered notes are {len(tweets)}")

Total unique tweets scraped from filtered notes are 82155


In [3]:
# Year-wise stats of tweets
# This is very obious that from 2020 the number of tweets are increasing
# X's Community Notes (formarly known as Twitter's Birdwatch) was launched on January 25, 2021 as a pilot program.
dates_stats = []
for i in tweets:
    dates_stats.append(tweets[i]["createdAt"][-4:])

print("Tweets year-wise stats:")
print(pd.Series(dates_stats).value_counts().sort_index(ascending=False))

Tweets year-wise stats:
2025    22083
2024    41211
2023    13645
2022     2696
2021     2357
2020       72
2019       29
2018       17
2017       17
2016       14
2015        3
2014        2
2013        5
2012        1
2008        1
            2
Name: count, dtype: int64


In [4]:
# Checking if all tweets are in english & if not, poping them out
# This will also help in filtering link-only tweets where we don't have any text to analyze
# Also removing tweets older than 2021 (because of very few tweets before that)
non_en_yrs_cnt = 0
tweetIds = list(tweets)
for i in tweetIds:
    if tweets[i]["lang"] != "en":
        non_en_yrs_cnt += 1
        tweets.pop(i)
    elif tweets[i]["createdAt"][-4:] not in ["2021", "2022", "2023", "2024", "2025"]:
        non_en_yrs_cnt += 1
        tweets.pop(i)
        
print(f"Removed {non_en_yrs_cnt} non-english or older than 2021 tweets, out of {len(tweetIds)} total tweets.")
print(f"Remaining tweets: {len(tweets)}")

Removed 3457 non-english or older than 2021 tweets, out of 82155 total tweets.
Remaining tweets: 78698


In [5]:
# Saving tweets and notes (with ratings) dataset to "Data" folder
with open('../Data/tweets.json', 'w') as f:
    json.dump(tweets, f, indent=4)

df_notes = pd.read_csv("Outputs/notes.tsv", sep="\t", low_memory=False)
df_notes = df_notes[df_notes.tweetId.isin([int(i) for i in tweets.keys()])]

df_notes.to_csv("../Data/notes.tsv", sep="\t", index=False)

print(f"Unique tweetIds in notes data (now): {df_notes.tweetId.unique().shape[0]}")

Unique tweetIds in notes data (now): 78698


In [6]:
# train-test split (90-10) based on tweet creation date
def to_datetime(timestamp):
    return datetime.strptime(timestamp, "%a %b %d %H:%M:%S %z %Y")
    
sorted_tweets = dict(sorted(tweets.items(), key=lambda x: to_datetime(x[1]['createdAt'])))

tweetsId_split = list(sorted_tweets.keys())
train_size = int(len(tweetsId_split) * 0.9)

train_tweetIds = tweetsId_split[:train_size]
test_tweetIds = tweetsId_split[train_size:]

len(train_tweetIds), len(test_tweetIds)

(70828, 7870)

In [7]:
# Saving train and test tweetIds to json files
train_dic = {"tweetId": train_tweetIds}
test_dic = {"tweetId": test_tweetIds}

with open('../Data/train.json', 'w') as f1:
    json.dump(train_dic, f1, indent=4)

with open('../Data/test.json', 'w') as f2:
    json.dump(test_dic, f2, indent=4)

In [8]:
# Checking how many notes in test set have atleast one helpful note
test_tweets = [int(i) for i in test_dic["tweetId"]]

df_notes_helpful = df_notes[df_notes.helpfulnessStatus == "CURRENTLY_RATED_HELPFUL"]
df_test_notes_helpful = df_notes_helpful[df_notes_helpful.tweetId.isin(test_tweets)]

# printing no of test set notes and unique tweets in those notes
len(df_test_notes_helpful.noteId.unique()), len(df_test_notes_helpful.tweetId.unique())

(575, 488)

In [9]:
def timeToDate(createdAtMillis):
  # Convert to seconds
  seconds = createdAtMillis / 1000.0

  # Convert to UTC datetime
  dt_utc = datetime.fromtimestamp(seconds, UTC)

  # Format like "Wed Apr 24 16:31:13 +0000 2024"
  date = dt_utc.strftime("%a %b %d %H:%M:%S +0000 %Y")
  return date

# (1746088488687, 1757176897506) --> Thu May 01 2025 08:34:48.687; Sat Sep 06 2025 16:41:37.506
print("Test notes with helpful ratings date range (to ensure coverage):")
timeToDate(df_test_notes_helpful.createdAtMillis.min()), timeToDate(df_test_notes_helpful.createdAtMillis.max())

Test notes with helpful ratings date range (to ensure coverage):


('Thu May 01 08:34:48 +0000 2025', 'Sat Sep 06 16:41:37 +0000 2025')

In [10]:
# Saving test set tweets-notes with one helpful note to json file
df_test_notes_1_helpful = df_test_notes_helpful.drop_duplicates(subset=['tweetId'], keep='first')

helpful_test_dic = {"tweetId": df_test_notes_1_helpful.tweetId.astype(str).tolist()}

with open('../Data/test_helpful.json', 'w') as f3:
    json.dump(helpful_test_dic, f3, indent=4)